In [1]:
# Importing necessary libraries

import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import math
from scipy.spatial.transform import Rotation as R
from scipy.linalg import block_diag
import timeit as timee
#import import_ipynb
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
import matplotlib.collections
#import effort_solver_nn_3_triangles_3D_stiffness_information_Classical_FEM_changed as solv
#import Functions_for_material_law as ML
from sklearn.preprocessing  import MinMaxScaler
from datetime import datetime
start_time = datetime.now()
from mpl_toolkits.mplot3d import Axes3D
import sympy as sym
from matplotlib.patches import Rectangle
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Lambda, Dense, Conv2D, MaxPool2D, Dropout,LSTM,Embedding,Layer, LSTM, Conv1D, MaxPool1D, TimeDistributed, Input, GRU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import Callback
#import plotly.graph_objects as go
import pickle
import multiprocessing
#from numba import njit
from tensorflow.keras.models import Model, load_model
import keras_lmu
from tensorflow.keras.models import Model, load_model
from keras.layers.rnn.dropout_rnn_cell_mixin import DropoutRNNCellMixin
#from keras.layers.rnn.dropout_rnn_cell_mixin import DropoutRNNCellMixin
import tensorflow_addons as tfa
%matplotlib qt

# 2. Material model parameters and desired functions

In [2]:

def gen_input(stress_dev_trial, stress_back_trial):

    comp1, comp2, comp3, comp4, comp5, comp6                                             = stress_dev_trial[0][0], stress_dev_trial[1][1], stress_dev_trial[2][2], stress_dev_trial[1][2], stress_dev_trial[0][2], stress_dev_trial[0][1]
    compX1, compX2, compX3, compX4, compX5, compX6                                       = stress_back_trial[0][0], stress_back_trial[1][1], stress_back_trial[2][2], stress_back_trial[1][2], stress_back_trial[0][2], stress_back_trial[0][1]
    #comp1, comp2, comp3, comp4, comp5, comp6       = ds
    #compX1, compX2, compX3, compX4, compX5, compX6 = sb
    #int(comp1, comp2, comp3, comp4, comp5, comp6, compX1, compX2, compX3, compX4, compX5, compX6)
    x_batch_train                                  = np.array([[comp1, comp2, comp3, comp4, comp5, comp6, compX1, compX2, compX3, compX4, compX5, compX6]])/100
    x_batch_train                                  = tf.cast(abs(x_batch_train), dtype = tf.float32)
    x_batch_train                                  = tf.reshape(x_batch_train, shape=(1,1,12))
    
    return x_batch_train


def tens_to_voigt(tens):
    
    stress    = np.zeros(shape=(6,))
    
    stress[0] = tens[0][0]
    stress[1] = tens[1][1]
    stress[2] = tens[2][2]
    stress[3] = tens[1][2]
    stress[4] = tens[0][2]
    stress[5] = tens[0][1]
    
    return stress


def voigt_to_tens(stress):
    
    tens = np.zeros(shape=(3,3))
    
    
    tens[0][0] = stress[0]
    tens[1][1] = stress[1]
    tens[2][2] = stress[2]
    tens[0][1] = stress[5]
    tens[0][2] = stress[4]
    tens[1][0] = stress[5]
    tens[1][2] = stress[3]
    tens[2][0] = stress[4]
    tens[2][1] = stress[3]
    
    
    return tens


def props_update(mat_params, i,l,j,k, pois):
    
    return (mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]*pois)/((1 + pois)*(1 - 2*pois)), 0.5*mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 + pois)

def split(tens):
   
    I      = np.eye(3,3)
    
    aa     = voigt_to_tens(tens)
    
    shydro = (1/3*(aa.trace()))
    
    return aa - shydro*I, shydro*I


#......................................................................................................................................................

wahl                        = 0
height                      = 1
thick                       = 1
number_of_elements          = 1
DOF_per_node                = 5
theta                       = 0
pois                        = 0.32
nn                          = {}
nodes                       = {}
ax                          = {}
result                      = []
stiffness                   = []
N                           = 2.79
dummy                       = [] # This variable is used so that while doing df/dx we can find loop it for calculating
count                       = 0
NEWTON                      = 300
nodes_per_element           = 9
timeincrement               = 1e-4

gauss_points                = []
section_points              = []


# Initializing all the material properties

mat_params    = {}
I             = np.eye(3,3)
units         = 208


for i in range(1):
    
    for l in range(1):
    
        for j in range(1):

            for k in range(1):


                mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0 
                mat_params['stress_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['flow'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = np.zeros(shape=(3,3))
                mat_params['eps_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(3,3))
                mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3))
                mat_params['stress_n'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(6,))
                mat_params['stress_yield'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 110
                mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = 67400
                mat_params['delta_eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                         = np.zeros(shape=(6,))
                mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                     = np.zeros(shape=(6,))
                #mat_params['strain_n_1']                                                   = np.zeros(shape=(6,))
                mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]   = props_update(mat_params, i,l,j,k, pois) 
                mat_params['pois'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = pois
                mat_params['stress_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = np.zeros(shape=(6,))
                mat_params['eps_vec'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                           = np.zeros(shape=(6,))
                aaa, bbb = mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]
                aa = 2*bbb + aaa
                mat_params['stress_K'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = 3.42
                mat_params['abs_stress'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['stress_back'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.zeros(shape=(3,3))
                mat_params['abs_strain'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])
                mat_params['C_evp_fourth'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3,3,3))
                mat_params['C_evp'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                      = mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)].copy()

                mat_params['C_ep'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                                    = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])

                mat_params['EQPLAS'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.sqrt((np.tensordot(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)],mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]))*(2/3))
                K                                                                          = mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 - 2*pois)
                mat_params['Delta_d'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0
                mat_params['Delta_pl'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                 = 0
                mat_params['a_unit'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = 3100
                mat_params['s_unit'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = 110
                mat_params['Delta_epsilon_p'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]          = []
                mat_params['input'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                    = []
                mat_params['R'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 0.0
                mat_params['b1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 0.
                mat_params['b2'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 0.0
                mat_params['state_h_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['state_h1_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c1_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['state_h2_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c2_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['state_h3_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c3_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['seq_delta_eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]              = []
                
            mat_params['stiff_point'+str(i+1)+str(l+1)+str(j+1)]                           = np.zeros(shape=(DOF_per_node*nodes_per_element, DOF_per_node*nodes_per_element))     
    
    ax['element'+str(i+1)]                                                                 = np.zeros(shape=(DOF_per_node*nodes_per_element ,1))
    nn['element_force'+str(i+1)]                                              = []
    nn['element_disp'+str(i+1)]                                               = []
    nn['element_stiff'+str(i+1)]                                              = []





# 3. Loesung Verfahren

In [3]:
def implicit_integration_vp_gnl_new(mat_params, i, gauss_points, section_points, N, timeincrement, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n):
    

    Delta_pl_L  = 0

    #if ((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])<0:
        #print(i, l, m, n, stress_eqv_trial, mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], N)
    
    Delta_pl_U1 = np.maximum(0,(((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))*timeincrement)
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    stress_eqv_trial_n  = np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial)))
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))



    if Delta_pl_U2>Delta_pl_U1==0:

        Delta_pl_U = Delta_pl_U2
        #print('Delta_pl_U1 is chosen')

    else:
        Delta_pl_U = Delta_pl_U1 
        #print('Delta_pl_U2 is chosen')

    if Delta_pl_U1<0:
        print(Delta_pl_U1, 'Gaussian punkt: '+str(i)+str(l)+str(m)+str(n))

    #print(Delta_pl_L, Delta_pl_U1, Delta_pl_U2, Delta_pl_U)

    for q in range(5000):

        h_L                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_L/timeincrement)**(1/N) 
        Z_L                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + mat_params['s_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L))
        alpha_L                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['a_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(1 + mat_params['s_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_L)))*(Delta_pl_L/h_L)
        J_L                      =         np.sqrt(1.5*np.tensordot(Z_L, Z_L))
        RHS_L                    =         (1/alpha_L)*(J_L) - h_L

        h_U                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_U/timeincrement)**(1/N) 
        Z_U                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 110*Delta_pl_U))
        alpha_U                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['a_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(1 + mat_params['s_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_U)))*(Delta_pl_U/h_U)
        J_U                      =         np.sqrt(1.5*np.tensordot(Z_U, Z_U))
        RHS_U                    =         (1/alpha_U)*(J_U) - h_U

        Delta_pl_N               =         Delta_pl_U - (RHS_U*(Delta_pl_L-Delta_pl_U))/(RHS_L - RHS_U)
        #print(Delta_pl_N)

        h_N                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl_N/timeincrement)**(1/N) 
        Z_N                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 110*Delta_pl_N))
        alpha_N                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['a_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]/(1 + mat_params['s_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl_N)))*(Delta_pl_N/h_N)
        J_N                      =         np.sqrt(1.5*np.tensordot(Z_N, Z_N))
        RHS_N                    =         (1/alpha_N)*(J_N) - h_N


        #print(RHS_N)
        if abs(RHS_N)<1e-7:

            Delta_pl = Delta_pl_N
            #print(Delta_pl*timeincrement)
            #print('Your increment number '+str(k+1)+' has converged with '+str(q+1)+' iterations')

            break

        elif RHS_N*RHS_L<0:

            Delta_pl_L = Delta_pl_L
            Delta_pl_U = Delta_pl_N

        elif RHS_N*RHS_U<0:
            Delta_pl_L = Delta_pl_N
            Delta_pl_U = Delta_pl_U
            
    #mat_params['C_ep'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params    = material_jacobian_evp(mat_params, i, N, I, l, m, n, Delta_pl_N, alpha_N*h_N, index, stress_dev_trial, timeincrement, Z_N, h_N, alpha_N, J_N)
                
                
    return Delta_pl_N, alpha_N, h_N, mat_params, (Z_N/J_N), Z_N, J_N





In [4]:
l = 0
m = 0
n = 0
random   = 0
Delta_pl = 0
wahl    = 0
deqpl   = []
deqpl1           = []
deqpl2           = []
eps33_ana        = []
eps33_ana_pl     = []
count_tr         = 0
sequence_tracker = 0
trial_stress = []
back_stress  = []
number_of_iteration = 0
start_time = datetime.now()
iterations    = []

increment_s = [18]

sign   =  [1]

for ll in range(len(sign)):
    
    print('......................................................................................................')
    print('Load Sequence '+str(l+1)+' The sign is '+str(sign[l]))
    print('......................................................................................................')


    for t in range(increment_s[ll]):

        mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = np.array([ 1.39385621e-04, -4.35790497e-07, -6.53881554e-05, -1.20143493e-06,
       -6.00364241e-06,  8.29015628e-07])    
    
    
        for l in range(1):
            
                for m in range(1):
        
                    for n in range(1):
        
                        mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]      = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                        sig_tr_i_1                                                        = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
                        #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                        stress_back_trial                                                 = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                        #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        stress_dev_trial,_                                                = split(sig_tr_i_1)
                        #if i==0 and m==0 and n==0:
                        #    print(mat_params['stress_n111'])
                        stress_eqv_trial                                                  = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                        N_dirr                                                            = ((stress_dev_trial - stress_back_trial)/stress_eqv_trial)
                        #if i==1 and l==1 and m==0 and n==0:
                        #    print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], 'before entering viscoplasticity: '+str(i)+str(l)+str(m)+str(n))
        
                        # Write a function to give the current yield stress
                        #mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = Uhard(mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        #print(mat_params)
        
                        if stress_eqv_trial > abs(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]):
        
                            #if i==1 and l==1 and m==0 and n==0:
                            #    print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], 'after entering viscoplasticity: '+str(i)+str(l)+str(m)+str(n), 'Equivalent stress: '+str(stress_eqv_trial))
        
                            for p in range(100):
        
                                if not(p==0):
        
                                    #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                    #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                                    #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                    #stress_dev_trial                                              = mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
                                    #if i==0 and m==0 and n==0:
                                    
                                    #    print(mat_params['stress_n111'])
                                    sig_tr_i_1                                                    = mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                    stress_dev_trial,_                                            = split(sig_tr_i_1)
                                    stress_eqv_trial                                              = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                                    sigma_33_trial                                                = sig_tr_i_1[2]
                                    if stress_eqv_trial>mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]:
                                        Delta_pl, alpha_N, h_N, mat_params,N_plas, Z_N, J_N           = implicit_integration_vp_gnl_new(mat_params, i, gauss_points, section_points, N, timeincrement, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n)
                                    else:
                                        Delta_pl=0
                                        N_plas  = np.zeros(shape=(3,3))
                                        
                                    sigma_i_1                                                     = sigma_33_trial - ((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl))*N_plas[2][2]
                                    delta_eps33_i_1                                               = delta_eps33_i_minus_1 -(((sigma_i)*(delta_eps33_i - delta_eps33_i_minus_1))/(1e-7 + sigma_i_1 - sigma_i))
                                    
                                    mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = delta_eps33_i_1
                                    delta_eps33_i_minus_1 =  delta_eps33_i
                                    delta_eps33_i         =  delta_eps33_i_1
                                    sigma_i               =  sigma_i_1
        
                                    #if i==1 and l==1 and m==0 and n==0:
                                    #    print(delta_eps33_i_minus_1, delta_eps33_i, 'After '+str(p+1)+' iteration step', 'after entering viscoplasticity: '+str(i)+str(l)+str(m)+str(n))
                                    
                                    if abs(delta_eps33_i - delta_eps33_i_minus_1)/abs(delta_eps33_i_1) < 1e-7:
                                        #plt.plot(mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                        print('Die Konvergenz erfolgt nach '+str(p+1)+' Iterationen für '+str(t+1)+' inkrement')   
                                        mat_params['input'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append([tens_to_voigt(stress_dev_trial), tens_to_voigt(stress_back_trial)])
                                        mat_params['Delta_epsilon_p'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(Delta_pl)
                                        break
                                    #print('Trial_Stress '+str(sigma_33_trial),'Sigma_i_1 ' +str(sigma_i_1), 'Equivalent Plastic Strain: '+str(Delta_pl))
                                    
                                    
                                    
                                    
                                elif p==0:
                            
                                    sigma_33_trial        = sig_tr_i_1[2]
                                    #mat_params, Delta_pl  = implicit_integration(mat_params, stress_eqv_trial,i,l,m,n)
                                    Delta_pl, alpha_N, h_N, mat_params, N_plas, Z_N, J_N  = implicit_integration_vp_gnl_new(mat_params, i, gauss_points, section_points, N, timeincrement, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n)
                                    sigma_i_1             = sigma_33_trial - ((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl))*N_plas[2][2]
                                    sigma_i               = sigma_i_1
                                    delta_eps33_i_minus_1 = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] 
                                    if mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0] == - mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1]:
                                        mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = 0.5*(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1])
                                        print('überraschungs fall')
                                    else:
                                        mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = -(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0] + mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1])
                            
                                    delta_eps33_i         = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2]
                                    
                                    #if i==1 and l==1 and m==0 and n==0:
                                    #    print(delta_eps33_i_minus_1, delta_eps33_i, 'After '+str(p+1)+' iteration step', str(i)+str(l)+str(m)+str(n))
                                    #print(delta_eps33_i, delta_eps33_i_minus_1, Delta_pl)
                            #Delta_pl    = 0
                            #mat_params['flag'+str(i+1)]+=1
                            #print(mat_params['flag'+str(i+1)])
                            #delta_ep    = 0
        
        #                     for j in range(1000):
        
        
        #                         RHS                              = stress_eqv_trial  - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] -  3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl))
        #                         #a                                = RHS/(mat_params['H'+str(i+1)] + (mat_params['E'+str(i+1)]))
        #                         #print(mat_params['H'+str(i+1)])
        #                         Delta_pl                         = Delta_pl +  (RHS/(mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + 3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
        #                         #delta_ep                         = delta_ep + (Delta_pl)
        #                         #print(Delta_pl)
        #                         #mat_params['EQPLAS'+str(i+1)]    = mat_params['EQPLAS'+str(i+1)] + Delta_pl
        
        #                         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = Uhard(mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]+Delta_pl)
        #                         #print(mat_params['H'+str(i+1)])
        #                         #print(Delta_pl, mat_params['EQPLAS'+str(i+1)], RHS)
        #                         #mat_params['C'+str(i+1)][0][0]                                = (mat_params['E'+str(i+1)]*mat_params['H'+str(i+1)])/(mat_params['E'+str(i+1)] + mat_params['H'+str(i+1)])
        #                         #print(RHS, mat_params['stress_yield'+str(i+1)])
        #                         if abs(stress_eqv_trial  - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] -  3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl)))<1e-5:
        #                             #print(j+1, stress_eqv_trial  - mat_params['stress_yield'+str(i+1)] -  (3*mat_params['mu'+str(i+1)]*(Delta_pl)))
        #                             #print('....Converged...... in '+str(j+1)+' iterations')
        #                             #print(Delta_pl)
        #                             break
        
                            # Now we move ahead with updating the stress, elastic and plastic strains
                            #print('...............out from the loop.......')
        
                            #if mat_params['flag'+str(i+1)]==0:
                            #    mat_params['stress_tens'+str(i+1)]        = ((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]) + I*(ss) 
                            #    print(mat_params['stress_tens'+str(i+1)])
                            #    mat_params['flag'+str(i+1)]               +=1
                            #    mat_params['stress_tens'+str(i+1)]        = ((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]) + I*(ss)  + np.array([[4.2,0,0],[0,0,0],[0,0,0]])
                            #    print(mat_params['stress_tens'+str(i+1)])
        
                            #else:
                            #if i==0 and m==0 and n==0:
        
                            #    print((mat_params['stress_n111'] - ML.tens_to_voigt(2*mat_params['mu111']*(1.5)*Delta_pl*(stress_dev_trial/(stress_eqv_trial))))[0])
                            #mat_params['stress_yield'+str(i+1)],_ = Uhard(mat_params['EQPLAS'+str(i+1)]+Delta_pl)
                            #dummy_sph                             = (stress_dev_trial/(stress_eqv_trial))
                            #mat_params['stress_yield'+str(i+1)],_ = Uhard(mat_params['EQPLAS'+str(i+1)]+Delta_pl)
                            #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = ((stress_dev_trial/(stress_eqv_trial))*(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])) + I*(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*dummy_sph[0][0]/(3*abs(dummy_sph[0][0])))
                            #mat_params['stress_tens'+str(i+1)]    = ((stress_dev_trial/(stress_eqv_trial))*(mat_params['stress_yield'+str(i+1)])) + I*(mat_params['stress_yield'+str(i+1)]/3)
                            #print(((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]))
        
                            #print('.............................')
                            #print()
                            #if i==1 and l==1 and m==0 and n==0:
                            #    print(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], 'after entering viscoplasticity: '+str(i)+str(l)+str(m)+str(n))
        
                            #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  = ML.tens_to_voigt(mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            a1                                        = tens_to_voigt(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            b1                                        = tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            c1                                        = tens_to_voigt(N_plas)
        
                            for j in range(3):
        
                                a1[j]                                 = a1[j]  +  (1.5*(Delta_pl)*(c1[j]/1))
                                b1[j]                                 = b1[j]  -  (1.5*(Delta_pl)*(c1[j]/1))
        
                            for j in range(3,6):
        
                                a1[j]                                 = a1[j]  +  (3*(Delta_pl)*(c1[j]/1))
                                b1[j]                                 = b1[j]  -  (3*(Delta_pl)*(c1[j]/1))
        
        
                            #mat_params['stress_n'+str(i+1)]           = mat_params['stress_n'+str(i+1)] + mat_params['C_ep'+str(i+1)].dot(mat_params['delta_eps'+str(i+1)]-1.5*(Delta_pl)*(c1/stress_eqv_trial))
        
                            mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = voigt_to_tens(a1)
                            mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = voigt_to_tens(b1)
                            mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]             = mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + Delta_pl
                            mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]               = tens_to_voigt(N_plas)
                            mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - (Delta_pl*mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                                          
                            #mat_params['flow'+str(i+1)]               = mat_params['flow'+str(i+1)].reshape(6,1)
                            mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]       = (mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['a_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl*N_plas))/(1 + mat_params['s_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl)
                            #print(mat_params['EQPLAS'+str(i+1)], Delta_pl)
                            # Updating the material jacobian with a function
                            mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]   = stress_eqv_trial
                            #mat_params['C_ep'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params    = material_jacobian_evp(mat_params, i, N, N_plas, I, l, m, n, Delta_pl, alpha_N*h_N, index, stress_dev_trial, timeincrement)
                            #mat_params                                                           = material_jacobian_evp(mat_params, i, N, N_plas, I, l, m, n, Delta_pl, J_N, index, stress_dev_trial, timeincrement, Z_N, h_N, alpha_N)
                            #mat_params    = material_jacobian_evp(mat_params, i, N, N_plas, I, l, m, n, Delta_pl, J_N, index, stress_dev_trial, timeincrement, Z_N, h_N, alpha_N)
                            mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                            mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append((mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                            mat_params['Delta_pl'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = Delta_pl
                            #print(mat_params['EQPLAS'+str(i+1)+str(m+1)+str(n+1)])
                            mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2]  = 0
                            mat_params['seq_delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            #print('Stress_yield through material law:',mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])
        
                            #print('....................................')
                            #print(mat_params['EQPLAS1'])
        
                        else:
        
                            mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]   = stress_eqv_trial
                            mat_params['C_ep'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]               = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                            mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                            mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                            mat_params['seq_delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])


......................................................................................................
Load Sequence 1 The sign is 1
......................................................................................................
Die Konvergenz erfolgt nach 5 Iterationen für 12 inkrement
Die Konvergenz erfolgt nach 5 Iterationen für 13 inkrement
Die Konvergenz erfolgt nach 5 Iterationen für 14 inkrement
Die Konvergenz erfolgt nach 5 Iterationen für 15 inkrement
Die Konvergenz erfolgt nach 5 Iterationen für 16 inkrement
Die Konvergenz erfolgt nach 5 Iterationen für 17 inkrement
Die Konvergenz erfolgt nach 5 Iterationen für 18 inkrement


In [5]:
plt.plot(mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])